# Star Wars Data Analysis

Este notebook realiza uma análise exploratória dos dados extraídos da SWAPI
(Star Wars API) e armazenados em PostgreSQL.

Objetivos da análise:

* Qual é o personagem que apareceu em mais filmes de Star Wars?
* Quais são os planetas mais quentes do universo de Star Wars?
* Quais são as naves espaciais mais rápida do universo de Star Wars?
* Qual é a arma mais poderosa do universo de Star Wars?

## Importar bibliotecas

In [30]:
import psycopg2
import pandas as pd
import os
import plotly.express as px
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [31]:
load_dotenv()

True

## Entendimento do problema

O universo de Star Wars possui uma grande quantidade de informações sobre personagens, planetas, naves espaciais e filmes, que podem ser exploradas para análise e geração de insights. Esses dados estão disponíveis publicamente através da Star Wars API (SWAPI), que fornece informações estruturadas sobre diversos elementos da franquia. No entanto, esses dados são disponibilizados em formato de API e de forma distribuída em diferentes endpoints, o que dificulta sua análise direta sem um processo de coleta, organização e armazenamento adequado.

A principal dificuldade para iniciar o projeto foi entender como estruturar um processo capaz de coletar esses dados da API, considerando que a SWAPI utiliza paginação e que as informações estão separadas em diferentes recursos, como people, planets, starships e films. Além disso, alguns campos retornados pela API podem conter valores inconsistentes como "unknown" ou "n/a", o que exige um processo de limpeza e padronização antes que os dados possam ser analisados de forma confiável.

Dessa forma, foi necessário desenvolver um pipeline de dados utilizando o processo ETL (Extract, Transform, Load) para automatizar a ingestão dessas informações. Inicialmente, os dados são extraídos da API utilizando Python, depois passam por uma etapa de transformação para tratamento e normalização dos valores, e por fim são carregados em um banco de dados PostgreSQL, executado em um ambiente containerizado com Docker. Após o armazenamento estruturado dos dados, ferramentas como Pandas e Jupyter Notebook são utilizadas para realizar análises exploratórias e responder perguntas sobre o universo de Star Wars, como a participação de personagens em filmes, características dos planetas e propriedades das naves espaciais.

## Conectar ao banco

In [32]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"), 
    port=os.getenv("DB_PORT")
)

## Análise exploratória

A análise exploratória tem como objetivo investigar e compreender melhor os dados que foram coletados e armazenados pelo pipeline de ETL. Nessa etapa, busca-se identificar padrões, características e possíveis relações entre as diferentes entidades do universo de Star Wars, como personagens, planetas, naves e filmes.

### Entendimento inicial das bases

In [33]:
query_inicial = """
SELECT 'people' AS table_name, COUNT(*) AS total_records FROM people
UNION ALL
SELECT 'planets', COUNT(*) FROM planets
UNION ALL
SELECT 'starships', COUNT(*) FROM starships
UNION ALL
SELECT 'films', COUNT(*) FROM films;
"""

df_counts = pd.read_sql(query_inicial, conn)
df_counts

,table_name,total_records
0,people,82
1,planets,60
2,starships,36
3,films,6


In [35]:
fig = px.bar(
    df_counts,
    x="table_name",
    y="total_records",
    title="Volume de dados ingeridos por tabelas",
    labels={
        "table_name": "Tabela",
        "total_records": "Quantidade de Registros"
    }
)

fig.show()

Como parte da análise inicial dos dados, foi realizada uma consulta para identificar a quantidade de registros em cada tabela da base obtida a partir da API SWAPI. Esse tipo de exploração inicial é importante para compreender a estrutura e o volume de dados disponíveis, permitindo ter uma visão geral do dataset antes de realizar análises mais específicas.

Os resultados mostram que a tabela people possui 82 registros, seguida pela tabela planets com 60 registros, starships com 36 registros e films com 6 registros. Esse resultado indica que o conjunto de dados possui uma maior quantidade de informações relacionadas a personagens e planetas, enquanto o número de filmes corresponde exatamente às produções principais da saga Star Wars presentes na API.

Esse entendimento inicial é importante porque ajuda a orientar as análises seguintes, permitindo identificar quais entidades possuem maior volume de dados e quais tabelas podem oferecer mais possibilidades de exploração dentro do universo da franquia.

### Qual é o personagem que apareceu em mais filmes de Star Wars?

In [17]:
query = """
SELECT name, films_count
FROM people
ORDER BY films_count DESC
LIMIT 5
"""

df_top_characters = pd.read_sql(query, conn)
df_top_characters

,name,films_count
0,Obi-Wan Kenobi,6
1,R2-D2,6
2,C-3PO,6
3,Yoda,5
4,Palpatine,5


In [18]:
fig = px.pie(
    df_top_characters,
    names="name",
    values="films_count",
    title="Participação dos personagens nos filmes"
)

fig.show()

Para identificar quais personagens aparecem em mais filmes do universo de Star Wars, foi realizada uma consulta na tabela people, ordenando os registros pelo atributo films_count, que representa a quantidade de filmes em que cada personagem aparece segundo os dados da API SWAPI.

Os resultados mostram que Obi-Wan Kenobi, R2-D2 e C-3PO lideram o ranking, cada um com participação em 6 filmes. Em seguida aparecem Yoda e Palpatine, ambos com 5 aparições.

Um insight interessante é que os personagens com maior número de participações são figuras centrais que atravessam diferentes momentos da narrativa da saga. No caso de R2-D2 e C-3PO, por exemplo, eles acompanham diversos protagonistas ao longo da história, o que contribui para sua presença recorrente nos filmes. Isso indica que esses personagens desempenham um papel importante na continuidade da narrativa e na conexão entre diferentes trilogias da franquia.

**Importante:** Embora algumas fontes indiquem que C-3PO aparece em cerca de 10 filmes da franquia Star Wars, os dados obtidos pela API SWAPI apontam 6 participações. Isso ocorre porque a API considera apenas os filmes listados em seu conjunto de dados principal, enquanto outras fontes incluem participações adicionais em outros filmes da franquia, como produções derivadas, animações ou aparições especiais. Dessa forma, a diferença está relacionada ao escopo dos dados utilizados na análise.

In [19]:
query_filmes = """
SELECT 
    episode_id AS episodio, 
    title AS titulo, 
    release_date AS lancamento
FROM films
ORDER BY episode_id;
"""

pd.read_sql(query_filmes, conn)

,episodio,titulo,lancamento
0,1,The Phantom Menace,1999-05-19
1,2,Attack of the Clones,2002-05-16
2,3,Revenge of the Sith,2005-05-19
3,4,A New Hope,1977-05-25
4,5,The Empire Strikes Back,1980-05-17
5,6,Return of the Jedi,1983-05-25


Através da tabela de filmes é possível saber a quantidade de filmes que foram extraidos da API e quais são esses filmes.

### Quais são os planetas mais quentes do universo de Star Wars?

In [20]:
query_planetas_quentes = """
SELECT 
    name, 
    climate, 
    terrain
FROM planets
WHERE climate LIKE '%arid%' 
   OR climate LIKE '%hot%' 
   OR climate LIKE '%tropical%'
ORDER BY climate ASC;
"""

df_hot_planets = pd.read_sql(query_planetas_quentes, conn)
df_hot_planets

,name,climate,terrain
0,Trandosha,arid,"mountains, seas, grasslands, deserts"
1,Tatooine,arid,desert
2,Socorro,arid,"deserts, mountains"
3,Iktotch,"arid, rocky, windy",rocky
4,Kalee,"arid, temperate, tropical","rainforests, cliffs, canyons, seas"
5,Malastare,"arid, temperate, tropical","swamps, deserts, jungles, mountains"
6,Mustafar,hot,"volcanoes, lava rivers, mountains, caves"
7,Saleucami,hot,"caves, desert, mountains, volcanoes"
8,Rodia,hot,"jungles, oceans, urban, swamps"
9,Felucia,"hot, humid",fungus forests


Para identificar os planetas mais quentes do universo de Star Wars, foi realizada uma consulta na tabela de planetas filtrando registros cujo atributo climate contém termos associados a altas temperaturas, como arid, hot e tropical. Esse critério foi utilizado porque a API SWAPI não possui uma variável numérica de temperatura, sendo necessário interpretar os tipos de clima como um indicativo das condições ambientais de cada planeta.

A partir dessa filtragem, foram identificados planetas como Tatooine, Mustafar, Rodia, Felucia e Saleucami, entre outros. A presença de climas como hot e arid indica ambientes caracterizados por altas temperaturas, desertos, atividade vulcânica ou regiões tropicais.

Um insight interessante é que os planetas classificados como mais quentes apresentam terrenos bastante variados, incluindo desertos, florestas tropicais, pântanos e regiões vulcânicas. Isso sugere que, mesmo dentro da categoria de climas quentes, existem diferentes tipos de ecossistemas extremos, como o ambiente desértico de Tatooine e o ambiente vulcânico de Mustafar, que é conhecido por rios de lava e intensa atividade geológica.

### Quais são as naves espaciais mais rápida do universo de Star Wars?

In [21]:
query_naves_rapidas = """
SELECT name, max_atmosphering_speed
FROM starships
WHERE max_atmosphering_speed IS NOT NULL
ORDER BY max_atmosphering_speed DESC
LIMIT 5
"""

df_fast_starships = pd.read_sql(query_naves_rapidas, conn)
df_fast_starships

,name,max_atmosphering_speed
0,H-type Nubian yacht,8000.0
1,Theta-class T-2c shuttle,2000.0
2,J-type diplomatic barge,2000.0
3,Solar Sailer,1600.0
4,Jedi Interceptor,1500.0


In [22]:
import plotly.express as px

fig = px.bar(
    df_fast_starships,
    x="name",
    y="max_atmosphering_speed",
    text="max_atmosphering_speed",
    title="Top 5 Naves Mais Rápidas do Universo Star Wars",
    labels={
        "name": "Nave",
        "max_atmosphering_speed": "Velocidade Máxima"
    }
)

fig.show()

Para identificar as naves espaciais mais rápidas do universo de Star Wars, foi realizada uma consulta na tabela starships, ordenando os registros pelo atributo max_atmosphering_speed, que representa a velocidade máxima que a nave pode atingir dentro da atmosfera de um planeta. Esse atributo foi utilizado como critério de análise por fornecer uma métrica direta de desempenho de velocidade entre as naves disponíveis na API SWAPI.

Os resultados indicam que a H-type Nubian yacht apresenta a maior velocidade registrada, com 8000, ficando significativamente acima das demais naves analisadas. Em seguida aparecem a Theta-class T-2c shuttle e a J-type diplomatic barge, ambas com velocidade máxima de 2000, seguidas pela Solar Sailer com 1600 e pela Jedi Interceptor com 1500.

Um insight interessante desse resultado é que nem sempre naves de combate são as mais rápidas, já que a nave com maior velocidade é um iate real. Isso sugere que, dentro do universo de Star Wars, diferentes tipos de naves podem priorizar desempenho em velocidade dependendo de sua função, como transporte diplomático, mobilidade estratégica ou deslocamento rápido entre sistemas.

### Qual é a arma mais poderosa do universo de Star Wars?

A API SWAPI não possui uma métrica direta para determinar qual é a arma mais poderosa do universo de Star Wars. No entanto, considerando o impacto narrativo e a capacidade de destruição apresentada nos filmes, a Death Star é geralmente considerada a arma mais poderosa da franquia Star Wars, pois possui a capacidade de destruir planetas inteiros com um único disparo. Dessa forma, mesmo que a API não forneça um campo específico para medir o poder das armas, a análise pode ser feita com base nas informações do universo da saga e na escala de destruição apresentada.

Além disso, essa análise evidencia a importância de um entendimento aprofundado do domínio do problema (negócio) ao trabalhar com dados. Nem sempre as bases de dados ou APIs fornecem métricas diretas para responder determinadas perguntas de negócio. Nesses casos, é fundamental que o engenheiro e analista compreenda o contexto do domínio analisado para definir critérios adequados de interpretação e complementar a análise com conhecimento do universo/problema que está sendo trabalhado.

Para este caso, a forma mais correta é considerar que as naves mais tecnologicas possuem as armas mais poderosas, e medir isso por:

* hyperdrive_rating

ou

* max_atmosphering_speed

O campo hyperdrive_rating torna-se o mais ideal pois representa a eficiência do sistema de viagem em hiperespaço das naves.

In [23]:
query_arma_poderosa = """
SELECT name, hyperdrive_rating
FROM starships
WHERE hyperdrive_rating IS NOT NULL
ORDER BY hyperdrive_rating DESC
LIMIT 5
"""

df_hyperdrive = pd.read_sql(query_arma_poderosa, conn)
df_hyperdrive

,name,hyperdrive_rating
0,Belbullab-22 starfighter,6.0
1,Rebel transport,4.0
2,Death Star,4.0
3,Slave 1,3.0
4,CR90 corvette,2.0


In [24]:
fig = px.scatter(
    df_hyperdrive,
    x="name",
    y="hyperdrive_rating",
    size="hyperdrive_rating",
    title="Naves com maior capacidade de hyperdrive",
    labels={
        "name": "Nave",
        "hyperdrive_rating": "Hyperdrive Rating"
    }
)

fig.show()

Ao ordenar as naves por esse atributo, foi possível identificar quais veículos apresentam os sistemas mais avançados segundo os dados disponíveis. Nesse resultado, a Belbullab-22 starfighter aparece com o maior rating (6.0), seguida por outras naves como Rebel transport, Death Star, Slave 1 e CR90 corvette.

Dessa forma, mesmo que o hyperdrive_rating não represente diretamente poder destrutivo, ele funciona como uma proxy tecnológica, permitindo realizar uma análise comparativa entre as naves e contribuir para a discussão sobre quais veículos poderiam representar maior capacidade tecnológica e, potencialmente, maior poder dentro do universo da franquia.

## Conclusão

A análise exploratória permitiu compreender melhor os dados coletados da API SWAPI e armazenados no banco de dados após a execução do pipeline ETL. A partir das consultas realizadas, foi possível validar os volumes de dados ingeridos e identificar algumas características relevantes do universo de Star Wars, como os personagens com maior número de aparições em filmes, planetas que apresentam climas mais extremos e as naves espaciais com maiores capacidades de velocidade e hyperdrive.

Além de gerar insights sobre os dados, essa etapa também foi importante para verificar a consistência das informações armazenadas e garantir que o processo de ingestão foi realizado corretamente. As visualizações geradas com Plotly auxiliaram na interpretação dos resultados de forma mais clara e interativa, facilitando a identificação de padrões e comparações entre as diferentes entidades analisadas. Dessa forma, a análise exploratória complementa o pipeline de dados desenvolvido, demonstrando como os dados coletados podem ser utilizados para gerar informações relevantes e apoiar análises futuras.